# Generate SFT Dataset

## Load Library

In [1]:
import os
import re
import json
import base64
import requests
import time
import mimetypes
import hashlib
from tqdm import tqdm
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv

## Configuration

In [2]:
GEN_TEST_WITH_MODEL = False

### Define Path

In [3]:
from pathlib import Path

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "metadata" / "mkn2_metadata.json").exists()
            and (candidate / "data").exists()
        ):
            return candidate

    # Fallback: assume notebook is inside MKN-2/rutilahu-vlm-etl/notebooks
    if start.name == "notebooks":
        return start.parents[1]

    return start.parent

PROJECT_ROOT = find_project_root()

METADATA_PATH = PROJECT_ROOT / "metadata" / "mkn2_metadata.json"
IMAGE_BASE_DIR = PROJECT_ROOT / "data"

# Output: MKN-2/data/sft_dataset
OUTPUT_BASE_DIR = PROJECT_ROOT / "data" / "sft_dataset_baru"

OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_JSONL = OUTPUT_BASE_DIR / "train.jsonl"
VAL_JSONL = OUTPUT_BASE_DIR / "val.jsonl"
TEST_JSONL = OUTPUT_BASE_DIR / "test.jsonl"
ALL_JSONL = OUTPUT_BASE_DIR / "all.jsonl"
CACHE_PATH = OUTPUT_BASE_DIR / "cache.json"

### OpenRouter

In [4]:
load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL_NAME = "openai/gpt-5-mini"

### OpenRouter Config

In [5]:
TEMPERATURE = 0.1
MAX_TOKENS = 10000
TIMEOUT = 120
MAX_RETRIES = 3
RETRY_SLEEP_SEC = 2
MAX_WORKERS = 15

In [6]:
print("MODEL_NAME:", MODEL_NAME)
print("OUTPUT_BASE_DIR:", OUTPUT_BASE_DIR)
print("METADATA_PATH:", METADATA_PATH)
print("GEN_TEST_WITH_MODEL:", GEN_TEST_WITH_MODEL)

MODEL_NAME: openai/gpt-5-mini
OUTPUT_BASE_DIR: /home/jovyan/work/MKN-2/data/sft_dataset_baru
METADATA_PATH: /home/jovyan/work/MKN-2/metadata/mkn2_metadata.json
GEN_TEST_WITH_MODEL: False


### OpenRouter Prompt

In [7]:
# ============================================================
# SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT_REASON = """
Kamu adalah ahli material bangunan yang bertugas menjelaskan ciri visual yang mendukung label aktual material bangunan.

TUGAS:
Amati foto rumah yang diberikan. Tulis deskripsi visual yang menjelaskan alasan mengapa komponen diberi label aktual tersebut. Jadi tugas kamu adalah menjustifikasi label aktualnya.

GUNAKAN CIRI VISUAL SEPERTI :
- tekstur
- pola
- bentuk permukaan
- susunan material
- karakter visual yang terlihat
- letak terlihatnya dimana

CONTOH GAYA BAHASANYA :
- terlihat menggunakan
- tampak tersusun dari
- memiliki pola
- memiliki permukaan
- tampak menggunakan

PENTING:
- Label aktual SUDAH ditentukan.
- Jangan melakukan klasifikasi ulang.
- Jangan mengubah label.
- jangan menyebut status.
- JANGAN BERHALUSINASi
- JANGAN menybutkan material lain yang tidak relevan dengan label aktual pada kalimat penjelasan.
- hindari kata kata yang abstrak dan tidak terlihat langsung secara visual
- jangan menjelaskan detail yang tidak relevan dengan kategori label aktual.
- jangan menjelaskan terkait kualitas rumah, tapi lebih ke ciri visual label aktualnya.
- Untuk label Tembok: Jika tertutup plester/cat, jelaskan permukaan bidang yang solid, rata, dan keras yang mengindikasikan konstruksi pasangan bata/batako permanen. Buat lebih konsisten penjelasan labelnya, tapi tetap mewakili visualnya.
- Untuk komponen atap jangan memperhatikan kanopi kalau atap utamanya masih kelihatan. Kalau atap utamanya tidak kelihatan maka label aktual pasti material kanopi. Sesuaikan dengan label aktual.
- Untuk komponen lantai jika lantai dilapisi karpet jangan mendeskripsikan karpetnya tapi cari terlebih dahulu area tepi yang mengekspos fondasi aslinya. Tapi kalau label aktualnya Parket/vinil/karpet barulah deskripsikan karpetnya.
- Jangan mengeluarkan output prediksi dan status. Keluarkan hanya penjelasan per komponen.

ATURAN:
- Tulis tepat SATU kalimat.
- Panjang penjelasan 12-20 kata.
- Jelaskan HANYA komponen yang sedang diproses. Jangan membahas objek lain seperti ventilasi, jendela, pintu.
- Abaikan objek selain komponen target.
- Anggap label aktual sudah benar.
- Material boleh disebut jika menjadi bagian dari alasan visual.
- Penjelasan visual harus ditekankan kenapa dia masuk ke kategori label itu, jelaskan ciri ciri visualnya dan terlihat dimana.
- Jangan gunakan tanda baca koma (,), titik dua(:), garis miring (/) dalam penjelasan, gunakan kata penghubung seperti "dan", "yang memiliki", "dengan", dst.

OUTPUT WAJIB JSON VALID:
OUTPUT HARUS BERUPA JSON valid dengan format berikut:
{
"atap":"Atap rumah terlihat menggunakan lembaran logam bergelombang pada bidang atap utama.",
"dinding":"Dinding luar rumah tampak memiliki permukaan bidang solid dan rata.",
"lantai":"Lantai dalam rumah terlihat menggunakan ubin keramik mengilap dengan pola kotak teratur dan garis nat yang jelas."
}

Jika Generate alasan = False:
isi dengan string kosong ("").
"""

# ============================================================
# DESKRIPSI SKEMA
# ============================================================

SCHEMA_DESC = {
    "A": "Exterior Only (foto tampak luar)",
    "B": "Interior Only (foto tampak dalam)",
    "C": "Multi-Image (foto tampak luar + tampak dalam)",
}


# ============================================================
# BUILD USER PROMPT
# ============================================================

def build_reason_user_text(
    house_id: str,
    schema: str,
    components: dict,
):
    component_text = ""

    for comp in ["atap", "dinding", "lantai"]:
        c = components[comp]

        component_text += f"""
{comp.capitalize()}
- Label aktual : {c["prediksi_label"]}
- Generate alasan : {c["needs_api"]}

"""

    user_prompt = f"""
Berikut data rumah yang harus dijelaskan.

HOUSE
- ID    : {house_id}
- Skema : {SCHEMA_DESC.get(schema, schema)}

KOMPONEN TARGET
{component_text}

TUGAS:

Label aktual SUDAH FINAL dan TIDAK BOLEH DIUBAH.

Untuk setiap komponen:
1. Gunakan Label aktual yang diberikan.
2. Cari ciri visual yang PALING MUNGKIN mendukung label tersebut.
3. Jangan menentukan material baru dari gambar.
4. Jika visual ambigu tetap pertahankan Label aktual.
5. Jangan mengganti label berdasarkan observasi.

Fokus hanya pada komponen target.
Abaikan objek lain pada gambar.

AWALI KALIMAT:
- Atap rumah ...
- Dinding luar rumah ...
- Lantai dalam rumah ...

ATURAN OUTPUT:
- Satu kalimat per komponen.
- Tiga komponen: Atap Dinding Lantai.
- Jangan menyebut status.
- Jangan melakukan klasifikasi ulang.
- Jangan menyebut material selain label aktual.
- Contoh bukan dasar klasifikasi.
- Hanya jelaskan ciri visual yang mendukung label aktual.
""".strip()
    return user_prompt

### SFT Prompt

In [8]:
SYSTEM_PROMPT_SFT = """Kamu adalah sistem AI inspeksi material bangunan untuk validasi data DTSEN. Tugasmu adalah menganalisis citra rumah secara visual untuk mengidentifikasi material terluas pada komponen bangunan, mengklasifikasikan jenis material yang teramati, membandingkannya dengan data referensi DTSEN, serta memberikan penjelasan berbasis bukti visual pada setiap komponen sebagai alasan hasil validasi.

Analisis komponen harus mengikuti instruksi tugas pada pesan user. Hanya lakukan prediksi pada komponen yang diizinkan untuk dianalisis sesuai skema input. Jika suatu komponen dinyatakan tidak dianalisis
pada instruksi user, maka komponen tersebut wajib mengikuti aturan yang ditetapkan user.

exterior = tampak luar
interior = tampak dalam

LABEL MATERIAL RESMI DTSEN :
Gunakan HANYA satu label per komponen dari daftar berikut. Jangan menggunakan label lain di luar daftar.

Komponen Atap:
Beton
Genteng
Seng
Asbes
Bambu
Kayu/sirap
Jerami/ijuk/daun-daunan/rumbia
Lainnya
Tidak terdeteksi

Komponen Dinding:
Tembok
Plesteran anyaman bambu/kawat
Kayu/papan/gypsum/GRC/calciboard
Anyaman bambu
Batang kayu
Bambu
Lainnya
Tidak terdeteksi

Komponen Lantai:
Marmer/granit
Keramik
Parket/vinil/karpet
Ubin/tegel/teraso
Kayu/papan
Semen/bata merah
Bambu
Tanah
Lainnya
Tidak terdeteksi

Jika komponen yang seharusnya dianalisis tidak terlihat, tertutup objek lain, berada di luar area citra, atau bukan visual rumah, gunakan:
Prediksi : Tidak terdeteksi
Status   : Tidak teridentifikasi
Alasan : (sesuaikan dengan komponen)
"Atap": "Variabel atap tidak dapat diidentifikasi karena komponen atap pada foto rumah tampak luar tidak terlihat."
"Dinding": "Variabel dinding tidak dapat diidentifikasi karena komponen dinding pada foto rumah tampak luar tidak terlihat."
"Lantai": "Variabel lantai tidak dapat diidentifikasi karena komponen lantai pada foto rumah tampak dalam tidak terlihat."

KETENTUAN STATUS :
Pilih tepat satu status untuk setiap komponen yang dianalisis:
Sesuai : hasil prediksi visual identik atau konsisten dengan data referensi DTSEN.
Tidak sesuai : prediksi visual berbeda dengan data referensi DTSEN.
Tidak teridentifikasi : digunakan ketika prediksi adalah "Tidak terdeteksi",yaitu saat komponen tidak terlihat, bukti visual tidak cukup, atau komponen tidak tersedia sesuai konteks citra dan instruksi user.

KETENTUAN ALASAN :
Untuk setiap komponen, hasilkan penjelasan visual untuk field Alasan:
- Komponen terdeteksi: Tulis minimal 10–20 kata yang menjelaskan karakteristik visual yang mendasari prediksi: tekstur, warna, pola, struktur permukaan, atau ciri material yang terlihat pada citra.
- Komponen tidak terdeteksi (Prediksi = "Tidak terdeteksi"): Gunakan kalimat baku yang tercantum pada instruksi user tanpa modifikasi apapun.

FORMAT OUTPUT :
Gunakan TEPAT format Toon berikut. Tidak ada teks lain di luar format ini.
Hasil[3]{Komponen,Prediksi,Status,Alasan}:
Atap,<label DTSEN>,<Status>,"<Alasan>"
Dinding,<label DTSEN>,<Status>,"<Alasan>"
Lantai,<label DTSEN>,<Status>,"<Alasan>"
"""

## SFT Dataset Generation Pipeline

### Load Metadata

In [9]:
if not METADATA_PATH.exists():
    raise FileNotFoundError(f"Metadata tidak ditemukan: {METADATA_PATH}")

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata_all = json.load(f)

records = [
    rec for rec in metadata_all
    if rec.get("split") in {"train", "val", "test"}
    and rec.get("images")
]

from collections import Counter
print("Total metadata:", len(metadata_all))
print("Total records dipakai:", len(records))
print("Split count:", Counter(rec.get("split") for rec in records))
print("House count:", Counter(rec.get("house_type") for rec in records))
print("Output format:", "TOON")

records_by_split = {
    split: [rec for rec in records if rec.get("split") == split]
    for split in ["train", "val", "test"]
}

for split, split_records in records_by_split.items():
    print(f"{split}: {len(split_records)}")

Total metadata: 5858
Total records dipakai: 5858
Split count: Counter({'train': 4667, 'val': 598, 'test': 593})
House count: Counter({'multi': 3365, 'single_exterior_only': 1253, 'single_interior_only': 1240})
Output format: TOON
train: 4667
val: 598
test: 593


### Helper Functions

In [10]:
REFUSAL_KEYWORDS = [
    "maaf",
    "sorry",
    "cannot",
    "can't",
    "tidak bisa membantu",
    "tidak dapat membantu",
    "i can't",
    "unable",
    "refuse",
    "refusal",
]

In [11]:
def load_cache(path: Path) -> Dict[str, dict]:
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [12]:
def save_cache(path: Path, cache: Dict[str, dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

In [13]:
# ============================================================
# PARSE ALASAN JSON (UNTUK ASSISTANT)
# ============================================================

def parse_reason_json(
    raw: str,
):

    try:

        data = json.loads(raw)

        return {
            "atap":
                data.get(
                    "atap",
                    "",
                ),

            "dinding":
                data.get(
                    "dinding",
                    "",
                ),

            "lantai":
                data.get(
                    "lantai",
                    "",
                ),
        }

    except Exception:

        return {
            "atap": "",
            "dinding": "",
            "lantai": "",
        }

In [14]:
# ============================================================
# STATUS (UNTUK ASSISTANT)
# ============================================================

def resolve_status(
    actual_label,
    dtsen,
):
    pred = str(actual_label).strip().lower()
    target = str(dtsen).strip().lower()

    if pred in ["", "Tidak terdeteksi", "none"]:
        return "Tidak sesuai"

    if pred == target:
        return "Sesuai"

    return "Tidak sesuai"

In [15]:
# ============================================================
# SKEMA & RESOLUSI KOMPONEN
# ============================================================

KALIMAT_SKEMA = {
    "lantai":
        "Variabel lantai tidak dapat diidentifikasi karena foto rumah tampak dalam tidak tersedia.",

    "atap":
        "Variabel atap tidak dapat diidentifikasi karena foto rumah tampak luar tidak tersedia.",

    "dinding":
        "Variabel dinding tidak dapat diidentifikasi karena foto rumah tampak luar tidak tersedia.",
}

KALIMAT_TIDAK_TERLIHAT = {
    "atap":
        "Variabel atap tidak dapat diidentifikasi karena komponen atap pada foto rumah tampak luar tidak terlihat.",

    "dinding":
        "Variabel dinding tidak dapat diidentifikasi karena komponen dinding pada foto rumah tampak luar tidak terlihat.",

    "lantai":
        "Variabel lantai tidak dapat diidentifikasi karena komponen lantai pada foto rumah tampak dalam tidak terlihat.",
}


def get_schema(
    images: list,
) -> str:

    view_types = {
        (
            img
            .get(
                "view_type",
                ""
            )
            .lower()
        )
        for img
        in images
    }

    if (
        "exterior"
        in view_types
        and
        "interior"
        in view_types
    ):
        return "C"

    if "exterior" in view_types:
        return "A"

    return "B"

In [16]:
def resolve_components(
    schema: str,
    actual_label: dict,
    status: dict,
):
    components = {}

    for comp in ["atap", "dinding", "lantai"]:
        label = (actual_label.get(comp) or "").strip()
        comp_status = (status.get(comp) or "").strip()

        can_detect = (
            (schema == "A" and comp in ["atap", "dinding"])
            or (schema == "B" and comp == "lantai")
            or (schema == "C")
        )

        if not can_detect:
            components[comp] = {
                "prediksi_label": "Tidak terdeteksi",
                "status": "Tidak teridentifikasi",
                "needs_api": False,
                "alasan_generated": KALIMAT_SKEMA[comp],
            }
            continue

        if label == "" or label.lower() == "tidak terdeteksi":
            components[comp] = {
                "prediksi_label": "Tidak terdeteksi",
                "status": "Tidak teridentifikasi",
                "needs_api": False,
                "alasan_generated": KALIMAT_TIDAK_TERLIHAT[comp],
            }
            continue

        components[comp] = {
            "prediksi_label": label,
            "status": comp_status or "Tidak sesuai",
            "needs_api": True,
            "alasan_generated": "",
        }

    return components

In [17]:
# ============================================================
# BUILD TOON
# ============================================================

def build_toon(
    components,
):

    rows = [
        "Hasil[3]{Komponen,Prediksi,Status,Alasan}:"
    ]

    mapping = [
        ("atap", "Atap"),
        ("dinding", "Dinding"),
        ("lantai", "Lantai"),
    ]

    for key, label in mapping:

        c = components[key]

        rows.append(
            f'{label},'
            f'{c["prediksi_label"]},'
            f'{c["status"]},'
            f'"{c["alasan"]}"'
        )

    return "\n".join(
        rows
    )

In [18]:
def is_valid_record_images(record: dict) -> bool:
    images = record.get("images") or []
    house_type = (record.get("house_type") or "").lower().strip()

    if house_type == "multi":
        has_ext = any((img.get("view_type") or "").lower().strip() == "exterior" for img in images)
        has_int = any((img.get("view_type") or "").lower().strip() == "interior" for img in images)
        return has_ext and has_int

    return len(images) >= 1

In [19]:
def image_path_from_metadata(image_path: str) -> Path:
    return IMAGE_BASE_DIR / Path(image_path)

In [20]:
def guess_mime_type(path: Path) -> str:
    ext = path.suffix.lower()
    if ext in {".jpg", ".jpeg"}:
        return "image/jpeg"
    if ext == ".png":
        return "image/png"
    if ext == ".webp":
        return "image/webp"
    mime, _ = mimetypes.guess_type(str(path))
    return mime or "image/jpeg"

In [21]:
def encode_image_to_data_url(path: Path) -> str:
    with open(path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{guess_mime_type(path)};base64,{encoded}"

In [22]:
def sample_key(record: dict) -> str:
    payload = {
        "house_id": record.get("house_id"),
        "house_type": record.get("house_type"),
        "split": record.get("split"),
        "images": [
            (img.get("image_id"), img.get("image_path"), img.get("view_type"))
            for img in (record.get("images") or [])
        ],
        "actual_label": record.get("actual_label", {}),
        "status": record.get("status", {}),
    }
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()

In [23]:
def select_images(record: dict) -> Tuple[Optional[dict], Optional[dict], Optional[dict]]:
    images = record.get("images") or []
    house_type = (record.get("house_type") or "").lower().strip()

    ext = None
    intr = None
    single = None

    if house_type == "multi":
        for img in images:
            vt = (img.get("view_type") or "").lower().strip()
            if vt == "exterior":
                ext = img
            elif vt == "interior":
                intr = img
    else:
        single = images[0] if images else None

    return ext, intr, single


In [24]:
def build_user_prompt_text(
    record: dict,
) -> str:
    house_id = record.get("house_id", "")
    dtsen = record.get("dtsen", {}) or {}
    images = record.get("images") or []

    schema = get_schema(images)

    if schema == "A":
        view_type = "exterior"
    elif schema == "B":
        view_type = "interior"
    else:
        view_type = "exterior + interior"

    atap = dtsen.get("atap", "-")
    dinding = dtsen.get("dinding", "-")
    lantai = dtsen.get("lantai", "-")
    header = f"""
Berikut data rumah yang harus divalidasi.

House ID : {house_id}
View Type : {view_type}

DATA REFERENSI DTSEN :

Atap : {atap}
Dinding : {dinding}
Lantai : {lantai}
""".strip()

    if schema == "A":
        body = """
INSTRUKSI TUGAS :

ANALISIS KOMPONEN YANG WAJIB DIANALISIS:
Lakukan analisis material bangunan HANYA pada komponen berikut:
- Atap : Identifikasi label material atap dominan atau material dengan luas area terbesar yang menutupi bangunan. Gunakan atap segitiga (atap utama) sebagai titik acuan. Jika terdapat lembaran atap tambahan yang berada di depan dinding segitiga (seperti kanopi, teras, pelindung masuk, atau atap tambahan lain), ABAIKAN material atap tersebut selama atap utama masih terlihat. Hanya gunakan material kanopi jika atap utama benar-benar tidak terlihat.
- Dinding : Identifikasi label material dinding dominan atau material dengan luas area terbesar yang terlihat pada tampak luar (fasad utama).

Untuk setiap komponen yang dianalisis di atas:
- Tentukan hasil Prediksi per komponen menggunakan label material resmi DTSEN.
- Tentukan Status (Sesuai / Tidak sesuai / Tidak teridentifikasi): Bandingkan hasil prediksi dengan DATA REFERENSI DTSEN dan tentukan status setiap komponen.
- Berikan penjelasan visual yang mendukung hasil prediksi untuk digunakan pada field Alasan.

KOMPONEN YANG TIDAK DIANALISIS:
Komponen Lantai TIDAK BOLEH DIANALISIS. Karena variabel lantai pada DTSEN mengacu pada lantai utama di bagian dalam rumah, sedangkan input hanya berupa citra exterior.
Gunakan nilai berikut secara WAJIB untuk komponen lantai :
Prediksi : Tidak terdeteksi
Status : Tidak teridentifikasi
Alasan : "Variabel lantai tidak dapat diidentifikasi karena foto rumah tampak dalam tidak tersedia."

Output berupa format output Toon
"""
    elif schema == "B":
        body = """
INSTRUKSI TUGAS :

ANALISIS KOMPONEN YANG WAJIB DIANALISIS:
Lakukan analisis material bangunan HANYA pada komponen lantai. Tentukan material lantai utama di dalam rumah berdasarkan luas area terbesar yang terlihat. Jika lantai tertutup lapisan tambahan seperti karpet, identifikasi terlebih dahulu material dasar yang masih terlihat. Jika material dasar tidak dapat diamati, gunakan label Parket/vinil/karpet.

Untuk komponen yang dianalisis:
- Tentukan hasil Prediksi material lantai berdasarkan bukti visual pada citra menggunakan label material resmi DTSEN.
- Tentukan Status (Sesuai / Tidak sesuai / Tidak teridentifikasi): Bandingkan hasil prediksi dengan DATA REFERENSI DTSEN dan tentukan status pada komponen lantai.
- Berikan penjelasan visual yang mendukung hasil prediksi untuk digunakan pada field Alasan.

KOMPONEN YANG TIDAK DIANALISIS:
Komponen Atap dan Dinding TIDAK BOLEH DIANALISIS. Karena input hanya berupa citra interior, maka material atap dan material dinding luar bangunan tidak dapat ditentukan berdasarkan definisi variabel DTSEN.
Gunakan nilai berikut secara WAJIB untuk komponen Atap dan Dinding :
Prediksi : Tidak terdeteksi
Status : Tidak teridentifikasi
Alasan :
Atap → "Variabel atap tidak dapat diidentifikasi karena foto rumah tampak luar tidak tersedia."
Dinding → "Variabel dinding tidak dapat diidentifikasi karena foto rumah tampak luar tidak tersedia."

Output berupa format output Toon
"""
    else:
        body = """
INSTRUKSI TUGAS :
Gunakan seluruh gambar yang diberikan untuk membantu memahami struktur rumah dan lokasi komponen.

Untuk setiap komponen, lakukan identifikasi berdasarkan komponen fisik yang benar, dengan prioritas observasi berikut:
Atap: Tentukan material terluas atap utama bangunan. Prioritaskan observasi dari foto tampak luar. Jika atap utama tidak terlihat dari luar tetapi struktur atau material atap terlihat jelas dari foto tampak dalam, informasi interior boleh digunakan sebagai bukti tambahan. Abaikan kanopi atau atap tambahan jika atap utama masih terlihat.
Dinding: Tentukan material terluas dinding luar bangunan (fasad utama), bukan sekat interior, furnitur, pagar, dekorasi, atau elemen non-struktural. Pilih material berdasarkan luas area terlihat terbesar.
Lantai: Tentukan material terluas lantai utama di dalam rumah. Prioritaskan material dasar yang terlihat jelas, bukan karpet jika masih ada material dasar yang tampak.

Untuk setiap komponen:
- Tentukan hasil Prediksi per komponen menggunakan label material resmi DTSEN.
- Tentukan Status (Sesuai / Tidak sesuai / Tidak teridentifikasi): Bandingkan hasil prediksi dengan DATA REFERENSI DTSEN dan tentukan status setiap komponen.
- Berikan penjelasan visual yang mendukung hasil prediksi untuk digunakan pada field Alasan.

KETENTUAN OUTPUT:
- Gunakan format Toon.
- Jangan menambah komponen lain di luar Atap Dinding dan Lantai.
"""

    return "\n\n".join([header, body]).strip()

In [25]:
def build_user_content(
    record: dict,
    components: dict = None,
    target: str = "sft",
):

    if target not in {
        "openrouter",
        "sft",
    }:
        raise ValueError(
            "target harus 'openrouter' atau 'sft'."
        )

    house_type = (
        (
            record.get(
                "house_type"
            )
            or ""
        )
        .lower()
        .strip()
    )

    ext_img, int_img, single_img = (
        select_images(
            record
        )
    )

    content = []

    def add_image(
        img_obj: dict,
        caption: str,
    ):

        meta_path = img_obj["image_path"]

        img_path = image_path_from_metadata(
            meta_path
        )

        if target == "openrouter":

            content.append({
                "type":
                "image_url",

                "image_url": {
                    "url":
                    encode_image_to_data_url(
                        img_path
                    )
                }
            })

        else:

            content.append({
                "type": "image",
        
                "image":
                meta_path
            })

        content.append({
            "type":
            "text",

            "text":
            caption,
        })

    # =========================
    # IMAGE
    # =========================

    if house_type == "multi":

        if ext_img:
            add_image(
                ext_img,
                "Foto tampak luar."
            )

        if int_img:
            add_image(
                int_img,
                "Foto tampak dalam."
            )

    elif single_img:

        vt = (
            single_img
            .get(
                "view_type",
                "exterior"
            )
            .lower()
        )

        add_image(
            single_img,
            (
                "Foto tampak luar."
                if vt == "exterior"
                else
                "Foto tampak dalam."
            )
        )

    # =========================
    # PROMPT
    # =========================

    content.append({

        "type":
        "text",

        "text":
        (
            build_reason_user_text(
                house_id=record["house_id"],
        
                schema=get_schema(
                    record["images"]
                ),
        
                components=components,
        
            )
        
            if target == "openrouter"
        
            else
        
            build_user_prompt_text(
                record
            )
        )

    })

    return content

In [26]:
def build_messages_for_api(
    record: dict,
    components: dict,
) -> List[dict]:

    return [

        {
            "role":
            "system",

            "content":
            get_system_prompt(
                target="openrouter"
            ),
        },

        {
            "role":
            "user",

            "content":
            build_user_content(
                record=record,
                components=components,
                target="openrouter",
            ),
        },

    ]

In [27]:
def build_reason_user_content(
    record,
    components,
):

    return build_reason_user_text(
        house_id=record["house_id"],
        schema=get_schema(
            record["images"]
        ),
        components=components,
    )

In [28]:
def clean_assistant_response(
    text: Optional[str],
) -> tuple[
    Optional[dict],
    str,
]:

    if not text:

        return (
            None,
            "empty_response"
        )

    cleaned = (
        text
        .strip()
    )

    if cleaned.startswith(
        "```"
    ):

        cleaned = (
            cleaned
            .replace(
                "```json",
                ""
            )
            .replace(
                "```",
                ""
            )
            .strip()
        )

    lowered = (
        cleaned
        .lower()
    )

    if any(
        x in lowered
        for x
        in REFUSAL_KEYWORDS
    ):

        return (
            None,
            "refusal"
        )

    try:

        parsed = (
            parse_reason_json(
                cleaned
            )
        )

        return (
            parsed,
            "ok"
        )

    except Exception:

        return (
            None,
            "invalid_json"
        )

In [29]:
def get_system_prompt(
    target: str,
) -> str:

    target = target.lower()

    if target == "openrouter":
        return SYSTEM_PROMPT_REASON

    if target == "sft":
        return SYSTEM_PROMPT_SFT

    raise ValueError(
        f"target tidak dikenal: {target}"
    )

In [30]:
def build_sft_sample(
    record: dict,
    assistant_text: Optional[str] = None,
) -> dict:
    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": get_system_prompt("sft"),
                }
            ],
        },
        {
            "role": "user",
            "content": build_user_content(record, target="sft"),
        },
    ]

    if assistant_text is not None:
        messages.append({
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": assistant_text,
                }
            ],
        })

    return {
        "id": {
            "house_id": record.get("house_id"),
        },
        "messages": messages,
    }

In [31]:
# ════════════════════════════════════════════════════════════
# — PARSER JSON + TOON FORMATTER (FIX NESTED OUTPUT)
# ════════════════════════════════════════════════════════════

import json
import ast


def extract_alasan(value):

    """
    Ambil field alasan saja jika model
    mengembalikan:
    {
        prediksi: ...,
        status: ...,
        alasan: ...
    }
    """

    if value is None:
        return ""

    # kalau nested dict
    if isinstance(
        value,
        dict
    ):

        return str(
            value.get(
                "alasan",
                ""
            )
        ).strip()

    return str(
        value
    ).strip()


def parse_alasan(raw: str):

    if not raw:
        return None

    try:

        raw = str(
            raw
        ).strip()

        # hapus ```json
        if raw.startswith(
            "```"
        ):

            raw = (
                raw
                .replace(
                    "```json",
                    ""
                )
                .replace(
                    "```",
                    ""
                )
                .strip()
            )

        # ======================
        # parse JSON
        # ======================

        try:

            data = json.loads(
                raw
            )

        except:

            data = ast.literal_eval(
                raw
            )

        return {

            "atap":
                extract_alasan(
                    data.get(
                        "atap",
                        ""
                    )
                ),

            "dinding":
                extract_alasan(
                    data.get(
                        "dinding",
                        ""
                    )
                ),

            "lantai":
                extract_alasan(
                    data.get(
                        "lantai",
                        ""
                    )
                ),
        }

    except Exception as e:

        print(
            f"[WARN] gagal parse: {e}"
        )

        print(
            raw[:500]
        )

        return None


# ============================================================
# TOON FORMATTER
# ============================================================

def build_toon(
    components: dict
):

    lines = [
        "Hasil[3]{Komponen,Prediksi,Status,Alasan}:"
    ]

    for comp in [

        "atap",
        "dinding",
        "lantai",

    ]:

        c = components[
            comp
        ]

        alasan = (
            str(
                c.get(
                    "alasan_generated"
                )
                or c.get(
                    "kalimat_baku"
                )
                or ""
            )
            .replace(
                "\n",
                " "
            )
            .strip()
        )

        lines.append(

            f'{comp.capitalize()},'
            f'{c["prediksi_label"]},'
            f'{c["status"]},'
            f'"{alasan}"'

        )

    return "\n".join(
        lines
    )


print(
    "✓ Parser & Toon formatter siap."
)

✓ Parser & Toon formatter siap.


### OpenRouter Call

In [32]:
session = requests.Session()


def call_openrouter(
    record: dict,
    components: dict,
) -> Optional[dict]:

    payload = {

        "model":
        MODEL_NAME,

        "messages":
        build_messages_for_api(
            record,
            components,
        ),

        "temperature":
        TEMPERATURE,

        "max_tokens":
        MAX_TOKENS,

        "reasoning": {
            "effort":
            "minimal"
        },

    }

    headers = {

        "Authorization":
        f"Bearer {API_KEY}",

        "Content-Type":
        "application/json",

    }

    last_error = None

    for attempt in range(
        1,
        MAX_RETRIES + 1,
    ):

        try:

            response = (
                session.post(
                    OPENROUTER_URL,
                    headers=headers,
                    json=payload,
                    timeout=TIMEOUT,
                )
            )

            if response.status_code != 200:

                last_error = (
                    f"HTTP {response.status_code}"
                )

                time.sleep(
                    RETRY_SLEEP_SEC
                    *
                    attempt
                )

                continue

            data = (
                response.json()
            )

            choice = (
                data
                .get(
                    "choices",
                    [{}],
                )[0]
            )

            message = (
                choice
                .get(
                    "message",
                    {}
                )
            )

            raw = (
                message
                .get(
                    "content",
                    ""
                )
            )

            parsed, reason = (
                clean_assistant_response(
                    raw
                )
            )

            print({

                "house_id":
                record.get(
                    "house_id"
                ),

                "status":
                reason,

                "success":
                parsed
                is not None,

            })

            if parsed:

                return (
                    parsed
                )

            last_error = (
                reason
            )

        except Exception as e:

            last_error = (
                str(
                    e
                )
            )

        time.sleep(
            RETRY_SLEEP_SEC
            *
            attempt
        )

    print(
        f"[ERROR] "
        f"house_id="
        f"{record.get('house_id')} | "
        f"{last_error}"
    )

    return None

### Generation Pipeline

In [33]:
def _generate_one_record(
    record: dict,
) -> dict:
    try:
        if not is_valid_record_images(record):
            return {
                "status": "skip",
                "reason": "invalid_images",
                "record": record,
                "assistant_text": None,
            }

        schema = get_schema(record.get("images", []))

        components = resolve_components(
            schema,
            record.get("actual_label", {}) or {},
            record.get("status", {}) or {},
        )

        raw_output = call_openrouter(
            record,
            components,
        )
        print("\n====================")
        print("HOUSE:", record.get("house_id"))
        print("RAW OPENROUTER:")
        print(raw_output)
        print("====================\n")

        if raw_output is None:
            return {
                "status": "fail",
                "reason": "openrouter_failed",
                "record": record,
                "assistant_text": None,
            }

        parsed = parse_alasan(raw_output)

        if parsed is None:
            print("[WARN] parse gagal:", record.get("house_id"))
            parsed = {}

        for comp in ["atap", "dinding", "lantai"]:
            if components[comp].get("needs_api"):
                value = parsed.get(comp, "")

                if isinstance(value, dict):
                    value = value.get("alasan", "")

                value = str(value).strip()
                components[comp]["alasan_generated"] = value

        assistant_text = build_toon(components)

        return {
            "status": "ok",
            "reason": None,
            "record": record,
            "assistant_text": assistant_text,
            "raw_reason_output": raw_output,
        }

    except Exception as e:
        print("[ERROR _generate_one_record]", record.get("house_id"), str(e))
        return {
            "status": "fail",
            "reason": str(e),
            "record": record,
            "assistant_text": None,
        }

In [34]:
def read_existing_ids(output_path: Path) -> set:
    if not output_path.exists():
        return set()

    done = set()

    with open(output_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                meta = obj["id"]
                done.add(meta.get("house_id"))
            except Exception:
                continue

    return done

In [35]:
def append_jsonl(
    path: Path,
    samples: List[dict],
) -> None:

    if not samples:
        return

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        path,
        "a",
        encoding="utf-8",
    ) as f:

        f.writelines(
            json.dumps(
                s,
                ensure_ascii=False,
            ) + "\n"
            for s in samples
        )

In [36]:
def generate_split(
    records: List[dict],
    split_name: str,
    output_path: Path,
    cache: Dict[str, dict],
    max_workers: int = MAX_WORKERS,
    use_openrouter: bool = True,
) -> List[dict]:
    done_ids = read_existing_ids(output_path)
    pending = []
    final_samples = []

    for record in records:
        record_id = record.get("house_id")

        if record_id in done_ids:
            continue

        if not is_valid_record_images(record):
            continue

        cache_key = sample_key(record)

        if (
            use_openrouter
            and cache_key in cache
            and cache[cache_key]["status"] == "ok"
        ):
            final_samples.append(
                build_sft_sample(
                    record,
                    assistant_text=cache[cache_key]["assistant_text"],
                )
            )
            continue

        pending.append(record)

    if use_openrouter and pending:
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {
                executor.submit(_generate_one_record, record): record
                for record in pending
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"{split_name}"
            ):
                result = future.result()

                if result is None:
                    continue

                record = result["record"]
                cache_key = sample_key(record)

                cache[cache_key] = {
                    "status": result["status"],
                    "reason": result["reason"],
                    "assistant_text": result["assistant_text"],
                }
                
                # simpan cache langsung
                save_cache(
                    CACHE_PATH,
                    cache,
                )
                
                if result["status"] == "ok":
                
                    sample = build_sft_sample(
                        record,
                        result["assistant_text"],
                    )
                
                    final_samples.append(
                        sample
                    )
                
                    # optional: simpan jsonl langsung
                    append_jsonl(
                        output_path,
                        [sample],
                    )
    else:
        for record in pending:
            final_samples.append(
                build_sft_sample(
                    record,
                    assistant_text=None,
                )
            )

    final_samples.sort(
        key=lambda x: (
            x["id"]["house_id"],
        )
    )

    append_jsonl(output_path, final_samples)
    return final_samples

## Run Generation

In [37]:
cache = load_cache(CACHE_PATH)

print(
    "Cache loaded:",
    len(cache)
)

# ==========================
# SORT
# ==========================

train_records = sorted(
    records_by_split["train"],
    key=lambda r: (
        r.get("house_type", ""),
        r.get("house_id", ""),
    ),
)

val_records = sorted(
    records_by_split["val"],
    key=lambda r: (
        r.get("house_type", ""),
        r.get("house_id", ""),
    ),
)

test_records = sorted(
    records_by_split["test"],
    key=lambda r: (
        r.get("house_type", ""),
        r.get("house_id", ""),
    ),
)

# ==========================
# GENERATE
# ==========================

train_samples = generate_split(
    train_records,
    "train",
    TRAIN_JSONL,
    cache,
    max_workers=MAX_WORKERS,
    use_openrouter=True,
)

save_cache(
    CACHE_PATH,
    cache,
)

val_samples = generate_split(
    val_records,
    "val",
    VAL_JSONL,
    cache,
    max_workers=MAX_WORKERS,
    use_openrouter=True,
)

save_cache(
    CACHE_PATH,
    cache,
)

test_samples = generate_split(
    test_records,
    "test",
    TEST_JSONL,
    cache,
    max_workers=MAX_WORKERS,
    use_openrouter=True,
)

save_cache(
    CACHE_PATH,
    cache,
)

# ==========================
# BUILD ALL
# ==========================

all_samples = []

for path in (
    TRAIN_JSONL,
    VAL_JSONL,
    TEST_JSONL,
):

    if not path.exists():
        continue

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        all_samples.extend(
            json.loads(line)
            for line in f
            if line.strip()
        )

with open(
    ALL_JSONL,
    "w",
    encoding="utf-8",
) as f:

    f.writelines(
        json.dumps(
            sample,
            ensure_ascii=False,
        )
        + "\n"
        for sample in all_samples
    )

# ==========================
# SUMMARY
# ==========================

print()

print(
    "Selesai."
)

print(
    "Train :",
    len(train_samples)
)

print(
    "Val   :",
    len(val_samples)
)

print(
    "Test  :",
    len(test_samples)
)

print(
    "All   :",
    len(all_samples)
)

print(
    "Cache :",
    len(cache)
)

print(
    "Output:",
    OUTPUT_BASE_DIR
)

Cache loaded: 0


train:   0%|          | 1/4667 [00:02<2:50:56,  2.20s/it]

{'house_id': 'H00035', 'status': 'ok', 'success': True}

HOUSE: H00035
RAW OPENROUTER:
{'atap': '', 'dinding': 'Dinding luar rumah tampak menggunakan permukaan bidang solid dan rata yang menunjukkan konstruksi pasangan tembok terlihat di bagian fasad.', 'lantai': 'Lantai dalam rumah terlihat menggunakan keramik dengan pola kotak teratur dan permukaan mengkilap yang terlihat di area ruang tamu.'}

{'house_id': 'H00047', 'status': 'ok', 'success': True}

HOUSE: H00047
RAW OPENROUTER:
{'atap': '', 'dinding': 'Dinding luar rumah tampak menggunakan permukaan bidang solid dan rata yang menunjukkan konstruksi pasangan tembok permanen.', 'lantai': 'Lantai dalam rumah terlihat menggunakan permukaan kasar dan padat yang menyerupai lapisan semen dengan warna abu dan tekstur bercak.'}

{'house_id': 'H00154', 'status': 'ok', 'success': True}

HOUSE: H00154
RAW OPENROUTER:
{'atap': '', 'dinding': 'Dinding luar rumah tampak memiliki permukaan bidang solid dan rata dengan tepian kusen yang menunjukkan

train:   0%|          | 8/4667 [00:02<16:18,  4.76it/s]  

{'house_id': 'H00032', 'status': 'ok', 'success': True}

HOUSE: H00032
RAW OPENROUTER:
{'atap': '', 'dinding': 'Dinding luar rumah tampak memiliki permukaan bidang solid dan rata dengan lapisan cat merata pada dinding fasad', 'lantai': ''}

{'house_id': 'H00040', 'status': 'ok', 'success': True}

HOUSE: H00040
RAW OPENROUTER:
{'atap': '', 'dinding': 'Dinding luar rumah tampak memiliki permukaan bidang solid dan rata dengan lapisan plester yang mengelupas dan bercak noda terlihat pada permukaannya.', 'lantai': 'Lantai dalam rumah terlihat berupa permukaan tanah padat yang tampak kasar dan tidak beraturan serta menempel langsung pada dinding tanpa lempengan lantai terlihat.'}

{'house_id': 'H00124', 'status': 'ok', 'success': True}

HOUSE: H00124
RAW OPENROUTER:
{'atap': 'Atap rumah tampak menggunakan genteng dengan susunan bergelombang teratur pada bidang atap utama.', 'dinding': 'Dinding luar rumah tampak memiliki permukaan bidang solid dan rata yang menunjukkan konstruksi pasangan per

...

test: 100%|█████████▉| 592/593 [01:35<00:00,  5.32it/s]

{'house_id': 'H61693', 'status': 'ok', 'success': True}

HOUSE: H61693
RAW OPENROUTER:
{'atap': 'Atap rumah tidak terdeteksi dalam foto interior sehingga tidak terlihat bidang atap utama.', 'dinding': 'Dinding luar rumah tidak terdeteksi dalam foto interior sehingga tidak terlihat permukaan dinding utama.', 'lantai': 'Lantai dalam rumah terlihat menggunakan karpet dengan motif warna pudar dan pola persegi yang menutupi sebagian besar bidang lantai.'}

{'house_id': 'H61679', 'status': 'ok', 'success': True}

HOUSE: H61679
RAW OPENROUTER:
{'atap': ' ', 'dinding': ' ', 'lantai': 'Lantai dalam rumah tampak menggunakan parket atau vinyl dengan papan panjang bercorak serat kayu yang tersusun rapat dan sejajar.'}



test: 100%|██████████| 593/593 [01:35<00:00,  6.20it/s]

{'house_id': 'H61666', 'status': 'ok', 'success': True}

HOUSE: H61666
RAW OPENROUTER:
{'atap': '', 'dinding': '', 'lantai': 'Lantai dalam rumah tampak menggunakan papan kayu dengan garis sambungan memanjang dan butiran serat terlihat pada bidang lantai.'}




Selesai.
Train : 4667
Val   : 598
Test  : 593
All   : 11716
Cache : 5858
Output: /home/jovyan/work/MKN-2/data/sft_dataset_baru


In [38]:
import requests
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

response = requests.get(
    "https://openrouter.ai/api/v1/credits",
    headers={
        "Authorization": f"Bearer {api_key}"
    }
)

data = response.json()["data"]

remaining = data["total_credits"] - data["total_usage"]

print(f"Total Credits : ${data['total_credits']:.4f}")
print(f"Total Usage   : ${data['total_usage']:.4f}")
print(f"Remaining     : ${remaining:.4f}")

Total Credits : $1728.0000
Total Usage   : $1542.2426
Remaining     : $185.7574


In [39]:
print(response.status_code)
print(response.text)

200
{"data":{"total_credits":1728,"total_usage":1542.242622095}}


In [40]:
import requests

response = requests.get(
    "https://openrouter.ai/api/v1/key",
    headers={"Authorization": f"Bearer {API_KEY}"}
)

print(response.json())

{'data': {'label': 'sk-or-v1-87c...593', 'is_management_key': False, 'is_provisioning_key': False, 'limit': 35, 'limit_reset': None, 'limit_remaining': 27.95050935, 'include_byok_in_limit': False, 'usage': 7.04949065, 'usage_daily': 7.04949065, 'usage_weekly': 7.04949065, 'usage_monthly': 7.04949065, 'byok_usage': 0, 'byok_usage_daily': 0, 'byok_usage_weekly': 0, 'byok_usage_monthly': 0, 'is_free_tier': False, 'expires_at': None, 'creator_user_id': 'user_353bjpWWM9WlfYoTzJJwHBQRbGP', 'rate_limit': {'requests': -1, 'interval': '10s', 'note': 'This field is deprecated and safe to ignore.'}}}
